# Federated Learning — Injury Prediction

This notebook demonstrates how **Federated Learning (FedAvg)** is applied to the injury-prediction Logistic Regression model from the companion notebook (`prediction-of-injury-with-logisticregression.ipynb`).

### Scenario
Multiple sports clubs each own a private player dataset. They want to jointly train a better injury-prediction model **without sharing raw player data**. Only model weights (coefficients + intercept) travel to the central server.

### FedAvg formula
$$\theta_{global} = \sum_{k=1}^{K} \frac{n_k}{n_{total}} \cdot \theta_k$$

Where $\theta_k$ are the local model weights of club $k$ and $n_k$ is the number of its players.

## 1. Imports & Config

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Simulation parameters
N_CLUBS    = 4     # number of sports clubs
FL_ROUNDS  = 15    # federated learning rounds
RANDOM_STATE = 42

## 2. Load & Preprocess Dataset

Same preprocessing as the original notebook:
- Drop features with low correlation to the target (Age, Height, Weight, BMI, Training Hours, Matches Played)
- Label-encode `Position`

In [ ]:
FEATURES = [
    'Position', 'Previous_Injury_Count', 'Knee_Strength_Score',
    'Hamstring_Flexibility', 'Reaction_Time_ms', 'Balance_Test_Score',
    'Sprint_Speed_10m_s', 'Agility_Score', 'Sleep_Hours_Per_Night',
    'Stress_Level_Score', 'Nutrition_Quality_Score', 'Warmup_Routine_Adherence',
]
TARGET = 'Injury_Next_Season'

df = pd.read_csv('data.csv')   # place the Kaggle dataset here
df = df.drop(columns=['Age', 'Matches_Played_Past_Season', 'Training_Hours_Per_Week',
                      'BMI', 'Height_cm', 'Weight_kg'])

le = LabelEncoder()
df['Position'] = le.fit_transform(df['Position'])

print(f'Dataset: {len(df)} players · {len(FEATURES)} features')
print(f'Injury rate: {df[TARGET].mean():.1%}')
df.head(3)

## 3. Centralized Baseline

Train a single Logistic Regression on **all** data — this is the upper-bound that FL tries to match without sharing raw data.

In [ ]:
X = df[FEATURES].values
y = df[TARGET].values

X_train_c, X_test, y_train_c, y_test = train_test_split(
    X, y, test_size=0.15, random_state=RANDOM_STATE
)

central_model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
central_model.fit(X_train_c, y_train_c)
baseline_acc = accuracy_score(y_test, central_model.predict(X_test))

print(f'Centralized accuracy: {baseline_acc:.5f}')
print()
print(classification_report(y_test, central_model.predict(X_test),
                             target_names=['No injury', 'Injury']))

## 4. Federated Learning Setup

Split the dataset into `N_CLUBS` partitions — each partition simulates a different sports club that **only sees its own players**.

In [ ]:
# Shared held-out test set (same as centralized — for fair comparison)
df_train, df_test = train_test_split(df, test_size=0.15, random_state=99)
X_test_fl = df_test[FEATURES].values
y_test_fl = df_test[TARGET].values

# Partition training data across clubs
club_dfs = np.array_split(df_train.sample(frac=1, random_state=RANDOM_STATE), N_CLUBS)

print('Club data distribution:')
for i, cdf in enumerate(club_dfs):
    print(f'  Club {i+1}: {len(cdf):4d} players  '
          f'injury rate: {cdf[TARGET].mean():.1%}  '
          f'(data stays on-premise — never shared)')

## 5. FedAvg Implementation

In [ ]:
def fed_avg(updates):
    """Weighted average of (coef, intercept) across clubs."""
    total = sum(n for _, _, n in updates)
    coef  = sum(c * n for c, _, n in updates) / total
    intercept = sum(b * n for _, b, n in updates) / total
    return coef, intercept


def local_train(club_df, global_coef, global_intercept):
    """
    Club trains locally using the global model as starting point.
    Returns updated (coef, intercept, n_samples).
    Raw data does NOT leave this function.
    """
    X_local = club_df[FEATURES].values
    y_local = club_df[TARGET].values

    model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, warm_start=True)
    model.coef_      = global_coef.copy()
    model.intercept_ = global_intercept.copy()
    model.classes_   = np.array([0, 1])
    model.fit(X_local, y_local)

    return model.coef_.copy(), model.intercept_.copy(), len(y_local)


print('FedAvg functions defined.')

## 6. Run FL Simulation

In [ ]:
# Initialise global model (zero weights)
global_coef      = np.zeros((1, len(FEATURES)))
global_intercept = np.zeros(1)

fl_accuracies = []

print(f"{'Round':<8} {'FL Accuracy':>12} {'vs Baseline':>14}")
print('-' * 36)

for round_n in range(1, FL_ROUNDS + 1):
    # Each club trains locally — no raw data shared
    updates = [
        local_train(club_df, global_coef, global_intercept)
        for club_df in club_dfs
    ]

    # Server aggregates with FedAvg
    global_coef, global_intercept = fed_avg(updates)

    # Evaluate global model on held-out test set
    eval_model = LogisticRegression(max_iter=1)
    eval_model.coef_      = global_coef
    eval_model.intercept_ = global_intercept
    eval_model.classes_   = np.array([0, 1])
    acc = accuracy_score(y_test_fl, eval_model.predict(X_test_fl))
    fl_accuracies.append(acc)

    delta = acc - baseline_acc
    print(f"{round_n:<8} {acc:>12.5f} {delta:>+14.5f}")

print(f'\nFinal FL accuracy : {fl_accuracies[-1]:.5f}')
print(f'Centralized       : {baseline_acc:.5f}')
print(f'Gap               : {fl_accuracies[-1] - baseline_acc:+.5f}')

## 7. Convergence Plot

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

rounds = list(range(1, FL_ROUNDS + 1))
ax.plot(rounds, fl_accuracies, 'o-', color='#6366f1', linewidth=2,
        markersize=5, label='FL global model (FedAvg)')
ax.axhline(baseline_acc, color='#10b981', linewidth=1.5,
           linestyle='--', label=f'Centralized baseline ({baseline_acc:.4f})')

ax.set_xlabel('FL Round', fontsize=11)
ax.set_ylabel('Accuracy', fontsize=11)
ax.set_title(f'Federated Learning Convergence ({N_CLUBS} clubs)', fontsize=13)
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.set_ylim(0.5, 1.02)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Feature Importance (Global Model)

In [ ]:
importance = np.abs(global_coef[0])
order      = np.argsort(importance)[::-1]

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(
    [FEATURES[i] for i in order],
    importance[order],
    color='#6366f1', alpha=0.8
)
ax.set_xlabel('|Coefficient| (global model)', fontsize=11)
ax.set_title('Feature Importance — FL Global Model', fontsize=13)
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Privacy Guarantee Summary

| What is shared | What stays local |
|---|---|
| Model coefficients (12 floats) | Raw player data |
| Model intercept (1 float) | Names, biometrics, injuries |
| Round accuracy (for logging) | Training logs, wellness data |

**FedAvg converges to accuracy comparable to centralized training** while each club retains full control of its player data — satisfying GDPR requirements.

## 10. Mapping to SportAnalytics Platform

| Model Feature | Platform Source (Sprint 3) |
|---|---|
| `Position` | `PlayerProfile.position` |
| `Previous_Injury_Count` | `COUNT(InjuryRecord)` |
| `Knee_Strength_Score` | `PhysicalAssessment.knee_strength_score` (latest) |
| `Hamstring_Flexibility` | `PhysicalAssessment.hamstring_flexibility` (latest) |
| `Reaction_Time_ms` | `PhysicalAssessment.reaction_time_ms` (latest) |
| `Balance_Test_Score` | `PhysicalAssessment.balance_test_score` *(to add)* |
| `Sprint_Speed_10m_s` | `PhysicalAssessment.sprint_speed_10m_s` *(to add)* |
| `Agility_Score` | `PhysicalAssessment.agility_score` *(to add)* |
| `Sleep_Hours_Per_Night` | `AVG(WellnessLog.sleep_hours)` |
| `Stress_Level_Score` | `AVG(WellnessLog.stress_level)` |
| `Nutrition_Quality_Score` | Derived from `WellnessLog` macros *(to add)* |
| `Warmup_Routine_Adherence` | `TrainingLog.warmup_adherence` *(to add)* |